In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt



In [ ]:
files = Path("./thresholds").glob("*.csv")
metodo = "EAR"
for file in files:
    if metodo in str(file):
        df = pd.read_csv(file)

threshold_columns = [col for col in df.columns if col not in ['trial', 'Should be']]

# Eliminar filas no numéricas en la columna 'trial' (como 'total')
df_clean = df[pd.to_numeric(df['trial'], errors='coerce').notnull()].copy()
df_clean['trial'] = df_clean['trial'].astype(int)

# Repetir el cálculo con el DataFrame limpio

# Resultados globales
results_global = []
for threshold in threshold_columns:
    preds = df_clean[threshold]
    truth = df_clean['Should be']
    mae = np.mean(np.abs(preds - truth))
    rmse = np.sqrt(np.mean((preds - truth) ** 2))
    results_global.append({'Threshold': threshold, 'Error Absoluto Medio': mae, 'Error Cuadrático Medio': rmse})

results_global_df = pd.DataFrame(results_global).sort_values(by='Error Cuadrático Medio')

# Resultados por trial
results_by_trial = {}
for _, row in df_clean.iterrows():
    trial_id = int(row['trial'])
    truth = row['Should be']
    trial_results = []
    for threshold in threshold_columns:
        pred = row[threshold]
        mae = abs(pred - truth)
        rmse = np.sqrt((pred - truth) ** 2)
        trial_results.append({'Threshold': threshold, 'Error Absoluto Medio': mae, 'Error Cuadrático Medio': rmse})
    results_by_trial[trial_id] = pd.DataFrame(trial_results).sort_values(by='Error Absoluto Medio')

results_global_df  # Mostrar mejores thresholds globales según MAE
##results_by_trial[6]



In [ ]:
# Calcular MAE y RMSE por threshold
results = []
for threshold in threshold_columns:
    preds = df_clean[threshold]
    truth = df_clean['Should be']
    mae = np.mean(np.abs(preds - truth))
    rmse = np.sqrt(np.mean((preds - truth) ** 2))
    results.append({
        'Threshold': threshold,
        'Threshold_float': float(str(threshold).replace(",", ".").replace(" ", "")),
        'MAE': mae,
        'RMSE': rmse
    })

results_df = pd.DataFrame(results).sort_values('Threshold_float')

# Graficar
plt.figure(figsize=(12, 6))
plt.plot(results_df['Threshold_float'], results_df['MAE'], label='EAM', marker='o')
plt.plot(results_df['Threshold_float'], results_df['RMSE'], label='ECM', marker='s')
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(metodo + " Error (EAM y ECM) vs. Threshold")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
results_df